# Handwriting OCR

This notebook uses Microsoft's TrOCR handwriting model to recognize handwritten lines. It detects text-line bands automatically, displays the proposed crops for review, then sends one crop at a time to TrOCR. The first model run downloads the pretrained weights.

In [ ]:
%pip install torch transformers pillow numpy

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps
from IPython.display import display

image_path = Path("handwriting_01.jpg")

image = Image.open(image_path).convert("RGB")
print(f"Image size: {image.width} x {image.height}")
display(image)

# Tune these if lines are missed or neighboring lines are joined.
ink_threshold = 180
minimum_ink_per_row = max(8, round(image.width * 0.01))
maximum_gap_between_rows = 6
vertical_padding = 12
horizontal_padding = 20

# Count dark pixels per row. Ignore tall vertical rules such as page borders.
ink_mask = np.asarray(ImageOps.grayscale(image)) < ink_threshold
vertical_rule_columns = ink_mask.sum(axis=0) > image.height * 0.08
ink_mask[:, vertical_rule_columns] = False
row_counts = ink_mask.sum(axis=1)
# Ignore scan-border rows that span most of the page width.
row_counts[row_counts > image.width * 0.5] = 0
active_rows = np.flatnonzero(row_counts >= minimum_ink_per_row)

# Join nearby active rows into line bands; the number of bands is data-driven.
line_bands = []
if active_rows.size:
    start = previous = int(active_rows[0])
    for row in active_rows[1:]:
        row = int(row)
        if row - previous > maximum_gap_between_rows + 1:
            line_bands.append((start, previous + 1))
            start = row
        previous = row
    line_bands.append((start, previous + 1))

line_images = {}
for top, bottom in line_bands:
    if bottom - top < 8 or row_counts[top:bottom].sum() < 25:
        continue
    minimum_ink_per_column = max(2, round((bottom - top) * 0.01))
    band_columns = np.flatnonzero(ink_mask[top:bottom].sum(axis=0) >= minimum_ink_per_column)
    if band_columns.size == 0:
        continue
    left = max(0, int(band_columns[0]) - horizontal_padding)
    right = min(image.width, int(band_columns[-1]) + horizontal_padding + 1)
    top = max(0, top - vertical_padding)
    bottom = min(image.height, bottom + vertical_padding)
    box = (left, top, right, bottom)
    label = f"Line {len(line_images) + 1}"
    crop = image.crop(box)
    line_images[label] = crop
    print(f"{label}: crop box {box}")
    display(crop)

if not line_images:
    raise ValueError("No text lines detected. Try lowering minimum_ink_per_row or ink_threshold.")


In [ ]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

model_name = "microsoft/trocr-base-handwritten"
processor = TrOCRProcessor.from_pretrained(model_name)
model = VisionEncoderDecoderModel.from_pretrained(model_name)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model.to(device)
print(f"Running on: {device}")

results = {}
for label, crop in line_images.items():
    pixel_values = processor(images=crop, return_tensors="pt").pixel_values.to(device)
    generated_ids = model.generate(pixel_values, max_new_tokens=128, num_beams=4)
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    results[label] = text
    print(f"{label}: {text}")

output_path = Path("handwriting_ocr.txt")
output_path.write_text("\n".join(f"{label}: {text}" for label, text in results.items()) + "\n", encoding="utf-8")
print(f"Saved recognized text to {output_path.resolve()}")
